Import

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import random

In [2]:
torch.manual_seed(42)

Prepare small dataset with sentence pair

In [3]:
pairs = [
    ("i love cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("i like cats", "j aime les chats"),
    ("i like dogs", "j aime les chiens"),
    ("i see cats", "je vois les chats"),
    ("i see dogs", "je vois les chiens"),
    ("you love cats", "tu aimes les chats"),
    ("you love dogs", "tu aimes les chiens"),
    ("you like cats", "tu aimes les chats"),
    ("you like dogs", "tu aimes les chiens"),
]

Build Vocabulary and also add aspecial tokens

In [4]:
def build_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }

    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab


src_vocab = build_vocab(
    [src for src, tgt in pairs]
)

tgt_vocab = build_vocab(
    [tgt for src, tgt in pairs]
)

print(src_vocab)
print(tgt_vocab)

{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'i': 4, 'love': 5, 'cats': 6, 'dogs': 7, 'like': 8, 'see': 9, 'you': 10}
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'j': 4, 'aime': 5, 'les': 6, 'chats': 7, 'chiens': 8, 'je': 9, 'vois': 10, 'tu': 11, 'aimes': 12}


Encode sequences

In [5]:
def encode_sentence(sentence, vocab):

    tokens = (
        ["<SOS>"]
        + sentence.split()
        + ["<EOS>"]
    )

    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]


src_sequences = [
    encode_sentence(src, src_vocab)
    for src, tgt in pairs
]

tgt_sequences = [
    encode_sentence(tgt, tgt_vocab)
    for src, tgt in pairs
]

src_tensor = torch.tensor(src_sequences)
tgt_tensor = torch.tensor(tgt_sequences)

print(src_tensor.shape)
print(tgt_tensor.shape)

torch.Size([10, 5])
torch.Size([10, 6])


Implement Positional Encoding

In [6]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_length=100):

        super().__init__()

        position = torch.arange(
            max_length
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) *
            (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(
            max_length,
            d_model
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # [1, max_length, d_model]

        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )

    def forward(self, x):

        sequence_length = x.size(1)

        return x + self.pe[:, :sequence_length]

Implementing attention computation

In [7]:
def scaled_dot_product_attention(
    Q,
    K,
    V,
    mask=None
):

    d_k = Q.size(-1)

    scores = (
        Q @ K.transpose(-2, -1)
    ) / math.sqrt(d_k)

    if mask is not None:

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

    attention_weights = torch.softmax(
        scores,
        dim=-1
    )

    output = attention_weights @ V

    return output, attention_weights

Implement the Multi Headed Attention

In [8]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads
    ):

        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads

        self.head_dim = (
            d_model // num_heads
        )

        self.W_Q = nn.Linear(
            d_model,
            d_model
        )

        self.W_K = nn.Linear(
            d_model,
            d_model
        )

        self.W_V = nn.Linear(
            d_model,
            d_model
        )

        self.W_O = nn.Linear(
            d_model,
            d_model
        )

    def forward(
        self,
        query,
        key,
        value,
        mask=None
    ):

        batch_size = query.size(0)

        # ----------------------------------
        # Linear projections
        # ----------------------------------

        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        # ----------------------------------
        # Split into multiple heads
        # ----------------------------------

        Q = Q.view(
            batch_size,
            -1,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            -1,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            -1,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Shapes:
        #
        # Q:
        # [batch, heads, seq, head_dim]
        #
        # K:
        # [batch, heads, seq, head_dim]
        #
        # V:
        # [batch, heads, seq, head_dim]

        # ----------------------------------
        # Attention
        # ----------------------------------

        output, attention_weights = (
            scaled_dot_product_attention(
                Q,
                K,
                V,
                mask
            )
        )

        # ----------------------------------
        # Combine heads
        # ----------------------------------

        output = output.transpose(
            1,
            2
        ).contiguous()

        output = output.view(
            batch_size,
            -1,
            self.d_model
        )

        # ----------------------------------
        # Final linear projection
        # ----------------------------------

        output = self.W_O(output)

        return output, attention_weights

Implement Feedforward layer on top of the transformers

In [9]:
class FeedForward(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                d_model,
                d_ff
            ),

            nn.ReLU(),

            nn.Linear(
                d_ff,
                d_model
            )
        )

    def forward(self, x):

        return self.network(x)

Implement the transformer encoder layer

In [10]:
class TransformerEncoderLayer(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff
    ):

        super().__init__()

        self.self_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        self.feed_forward = (
            FeedForward(
                d_model,
                d_ff
            )
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        x
    ):

        # ----------------------------------
        # Self-attention
        # ----------------------------------

        attention_output, attention_weights = (
            self.self_attention(
                x,
                x,
                x
            )
        )

        # Residual connection

        x = self.norm1(
            x + attention_output
        )

        # ----------------------------------
        # Feed-forward
        # ----------------------------------

        ff_output = self.feed_forward(x)

        # Residual connection

        x = self.norm2(
            x + ff_output
        )

        return x, attention_weights

Implement the transformers encoder (combining the 6 layer stacks)

In [12]:
class TransformerEncoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        max_length
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position = PositionalEncoding(
            d_model,
            max_length
        )

        self.layers = nn.ModuleList([

            TransformerEncoderLayer(
                d_model,
                num_heads,
                d_ff
            )

            for _ in range(num_layers)
        ])

    def forward(self, x):

        x = self.embedding(x)

        x = self.position(x)

        attention_weights = []

        for layer in self.layers:

            x, attention = layer(x)

            attention_weights.append(
                attention
            )

        return x, attention_weights

Implementing causal attention mask for autoregressive generation

In [13]:
def create_causal_mask(sequence_length):

    mask = torch.tril(
        torch.ones(
            sequence_length,
            sequence_length
        )
    )

    return mask

Implementing Transformers decoder layer

In [15]:
class TransformerDecoderLayer(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        d_ff
    ):

        super().__init__()

        self.self_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        self.cross_attention = (
            MultiHeadAttention(
                d_model,
                num_heads
            )
        )

        self.feed_forward = (
            FeedForward(
                d_model,
                d_ff
            )
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.norm3 = nn.LayerNorm(
            d_model
        )

    def forward(
        self,
        x,
        encoder_output,
        mask
    ):

        # ==================================
        # 1. MASKED SELF-ATTENTION
        # ==================================

        self_attention_output, self_attention_weights = (
            self.self_attention(
                x,
                x,
                x,
                mask
            )
        )

        x = self.norm1(
            x + self_attention_output
        )

        # ==================================
        # 2. CROSS-ATTENTION
        # ==================================

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                x,
                encoder_output,
                encoder_output
            )
        )

        x = self.norm2(
            x + cross_attention_output
        )

        # ==================================
        # 3. FEED-FORWARD
        # ==================================

        ff_output = self.feed_forward(x)

        x = self.norm3(
            x + ff_output
        )

        return (
            x,
            self_attention_weights,
            cross_attention_weights
        )

Implementing the transformers 6layer decoder level stacks

In [16]:
class TransformerDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        max_length
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position = PositionalEncoding(
            d_model,
            max_length
        )

        self.layers = nn.ModuleList([

            TransformerDecoderLayer(
                d_model,
                num_heads,
                d_ff
            )

            for _ in range(num_layers)
        ])

        self.output_layer = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(
        self,
        x,
        encoder_output,
        mask
    ):

        x = self.embedding(x)

        x = self.position(x)

        self_attention_weights = []
        cross_attention_weights = []

        for layer in self.layers:

            (
                x,
                self_attention,
                cross_attention
            ) = layer(
                x,
                encoder_output,
                mask
            )

            self_attention_weights.append(
                self_attention
            )

            cross_attention_weights.append(
                cross_attention
            )

        logits = self.output_layer(x)

        return (
            logits,
            self_attention_weights,
            cross_attention_weights
        )

Combining Transformers to a single class

In [17]:
class Transformer(nn.Module):

    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=128,
        num_heads=4,
        d_ff=512,
        num_layers=2,
        max_length=100
    ):

        super().__init__()

        self.encoder = TransformerEncoder(
            src_vocab_size,
            d_model,
            num_heads,
            d_ff,
            num_layers,
            max_length
        )

        self.decoder = TransformerDecoder(
            tgt_vocab_size,
            d_model,
            num_heads,
            d_ff,
            num_layers,
            max_length
        )

    def forward(
        self,
        source,
        target
    ):

        # ==================================
        # ENCODER
        # ==================================

        encoder_output, encoder_attention = (
            self.encoder(source)
        )

        # ==================================
        # DECODER
        # ==================================

        decoder_input = target[:, :-1]

        decoder_target = target[:, 1:]

        sequence_length = decoder_input.size(1)

        mask = create_causal_mask(
            sequence_length
        ).to(target.device)

        (
            logits,
            decoder_attention,
            cross_attention
        ) = self.decoder(
            decoder_input,
            encoder_output,
            mask
        )

        return (
            logits,
            decoder_target,
            encoder_attention,
            decoder_attention,
            cross_attention
        )

Create a transformers model

In [18]:
SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)

model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE,
    tgt_vocab_size=TGT_VOCAB_SIZE,

    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=2,
    max_length=20
)

print(model)

Transformer(
  (encoder): TransformerEncoder(
    (embedding): Embedding(11, 128)
    (position): PositionalEncoding()
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attention): MultiHeadAttention(
          (W_Q): Linear(in_features=128, out_features=128, bias=True)
          (W_K): Linear(in_features=128, out_features=128, bias=True)
          (W_V): Linear(in_features=128, out_features=128, bias=True)
          (W_O): Linear(in_features=128, out_features=128, bias=True)
        )
        (feed_forward): FeedForward(
          (network): Sequential(
            (0): Linear(in_features=128, out_features=512, bias=True)
            (1): ReLU()
            (2): Linear(in_features=512, out_features=128, bias=True)
          )
        )
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (decoder): TransformerDecoder(
    (embedding): Embedding(1

test model on forward pass

In [19]:
logits, targets, encoder_attn, decoder_attn, cross_attn = model(
    src_tensor,
    tgt_tensor
)

print("Logits:", logits.shape)
print("Targets:", targets.shape)

Logits: torch.Size([10, 5, 13])
Targets: torch.Size([10, 5])


train transformer model for a translstion task

In [20]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=tgt_vocab["<PAD>"]
)

EPOCHS = 1000

for epoch in range(EPOCHS):

    model.train()

    optimizer.zero_grad()

    (
        logits,
        targets,
        _,
        _,
        _
    ) = model(
        src_tensor,
        tgt_tensor
    )

    loss = criterion(
        logits.reshape(-1, TGT_VOCAB_SIZE),
        targets.reshape(-1)
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 100 == 0:

        print(
            f"Epoch {epoch+1}/{EPOCHS} "
            f"Loss: {loss.item():.4f}"
        )

Epoch 100/1000 Loss: 0.0027
Epoch 200/1000 Loss: 0.0014
Epoch 300/1000 Loss: 0.0008
Epoch 400/1000 Loss: 0.0006
Epoch 500/1000 Loss: 0.0004
Epoch 600/1000 Loss: 0.0003
Epoch 700/1000 Loss: 0.0002
Epoch 800/1000 Loss: 0.0002
Epoch 900/1000 Loss: 0.0002
Epoch 1000/1000 Loss: 0.0001


function for using trained model to translate

In [23]:
def translate(sentence, model, src_vocab, tgt_vocab, max_length=20):

    model.eval()

    # Encode the English input
    source_ids = encode_sentence(sentence, src_vocab)
    source = torch.tensor([source_ids], dtype=torch.long)

    # Reverse French vocabulary: ID -> word
    reverse_tgt_vocab = {
        idx: word for word, idx in tgt_vocab.items()
    }

    generated = [tgt_vocab["<SOS>"]]

    with torch.no_grad():

        # Encode source sentence once
        encoder_output, _ = model.encoder(source)

        # Generate one French token at a time
        for _ in range(max_length):

            decoder_input = torch.tensor(
                [generated], dtype=torch.long
            )

            # Causal mask for current target length
            seq_len = decoder_input.size(1)
            mask = create_causal_mask(seq_len)

            logits, _, _ = model.decoder(
                decoder_input,
                encoder_output,
                mask
            )

            # Predict the next token
            next_token = logits[0, -1, :].argmax().item()

            if next_token == tgt_vocab["<EOS>"]:
                break

            generated.append(next_token)

    # Convert token IDs to words
    words = [
        reverse_tgt_vocab.get(idx, "<UNK>")
        for idx in generated[1:]
    ]

    return " ".join(words)

In [24]:
sentence = "you see cats"

translation = translate(
    sentence,
    model,
    src_vocab,
    tgt_vocab
)

print("English:", sentence)
print("French:", translation)

English: you see cats
French: tu aimes les chats
